# Vacancies 2026 — Salary Prediction v3
**Target:** `salary_mean_net` | **Metric:** MAPE | **Models:** LightGBM + CatBoost + XGBoost ensemble

Improvements over v2:
- Smoothed Target Encoding (regularised by group size, no leakage)
- Employer-level aggregated stats: median, std, count, p25, p75
- Role/city interaction features
- XGBoost added to ensemble
- Larger SVD for description (100 components)

### 1. Imports & Config

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold
from sklearn.preprocessing import OrdinalEncoder
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor, Pool
import warnings
warnings.filterwarnings('ignore')

SEED        = 42
N_SPLITS    = 5
SVD_DESC    = 100
SVD_TITLE   = 40
TFIDF_DESC  = 6000
TFIDF_TITLE = 3000
# smoothed TE: weight of global mean = STE_K / (group_size + STE_K)
STE_K       = 20
np.random.seed(SEED)

### 2. Data Loading

In [2]:
train = pd.read_csv('data/train.csv')
test  = pd.read_csv('data/test_x.csv')
print(f'train: {train.shape}, test: {test.shape}')

train: (49051, 26), test: (12263, 25)


### 3. Text Feature Engineering (TF-IDF + SVD)

In [3]:
DESC_COL  = 'lemmaized_wo_stopwords_raw_description'
TITLE_COL = 'name_clean'

for df in [train, test]:
    df[DESC_COL]  = df[DESC_COL].fillna('')
    df[TITLE_COL] = df[TITLE_COL].fillna('')

n_train = len(train)
corpus_desc  = pd.concat([train[DESC_COL],  test[DESC_COL]],  ignore_index=True)
corpus_title = pd.concat([train[TITLE_COL], test[TITLE_COL]], ignore_index=True)

tfidf_desc  = TfidfVectorizer(max_features=TFIDF_DESC,  sublinear_tf=True, min_df=3, ngram_range=(1, 2), dtype=np.float32)
tfidf_title = TfidfVectorizer(max_features=TFIDF_TITLE, sublinear_tf=True, min_df=2, ngram_range=(1, 2), dtype=np.float32)
svd_desc    = TruncatedSVD(n_components=SVD_DESC,  random_state=SEED)
svd_title   = TruncatedSVD(n_components=SVD_TITLE, random_state=SEED)

mat_desc  = svd_desc.fit_transform(tfidf_desc.fit_transform(corpus_desc)).astype(np.float32)
mat_title = svd_title.fit_transform(tfidf_title.fit_transform(corpus_title)).astype(np.float32)

desc_cols  = [f'desc_svd_{i}'  for i in range(SVD_DESC)]
title_cols = [f'title_svd_{i}' for i in range(SVD_TITLE)]

svd_train = pd.DataFrame(np.hstack([mat_desc[:n_train], mat_title[:n_train]]), columns=desc_cols + title_cols)
svd_test  = pd.DataFrame(np.hstack([mat_desc[n_train:], mat_title[n_train:]]), columns=desc_cols + title_cols)

print(f'desc SVD var: {svd_desc.explained_variance_ratio_.sum():.3f} | title SVD var: {svd_title.explained_variance_ratio_.sum():.3f}')

desc SVD var: 0.235 | title SVD var: 0.272


### 4. Feature Engineering

In [4]:
EXPERIENCE_ORDER = ['Нет опыта', 'От 1 года до 3 лет', 'От 3 до 6 лет', 'Более 6 лет']

DROP_COLS = [
    'id', 'salary_mean_net',
    'raw_description',
    'raw_branded_description',
    'lemmaized_wo_stopwords_raw_branded_description',
    'lemmaized_wo_stopwords_raw_description',
    'name',
    'name_clean',
    'unified_address_country',  # 1 unique value
]

LOW_CARD_COLS = [
    'schedule_name', 'employment_name',
    'is_branded_description', 'if_foreign_language',
    'accept_handicapped', 'accept_kids',
]

# Cols for smoothed target encoding inside CV
TE_COLS = [
    'employer_id', 'employer_name',
    'professional_roles_name', 'specializations_profarea_name',
    'employer_industries',
    'unified_address_city', 'unified_address_state', 'unified_address_region',
]


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    exp_map = {v: i for i, v in enumerate(EXPERIENCE_ORDER)}
    df['experience_ord'] = df['experience_name'].map(exp_map).fillna(-1).astype(np.int8)

    df['desc_word_count'] = df[DESC_COL].str.split().str.len().fillna(0).astype(np.int16)

    df['has_skills']    = (df['key_skills_name'].fillna('[]') != '[]').astype(np.int8)
    df['has_languages'] = (df['languages_name'].fillna('[]') != '[]').astype(np.int8)
    df['skills_count']  = df['key_skills_name'].fillna('[]').str.count(',').astype(np.int16)

    city = df['unified_address_city'].fillna('').str.lower()
    df['is_moscow'] = city.str.contains('москва').astype(np.int8)
    df['is_spb']    = city.str.contains('санкт').astype(np.int8)

    for col in TE_COLS:
        if col in df.columns:
            df[col] = df[col].fillna('__NA__').astype(str)

    return df


train = engineer_features(train)
test  = engineer_features(test)

oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, dtype=np.float32)
train[LOW_CARD_COLS] = oe.fit_transform(train[LOW_CARD_COLS].astype(str))
test[LOW_CARD_COLS]  = oe.transform(test[LOW_CARD_COLS].astype(str))

TARGET = 'salary_mean_net'
y_raw  = train[TARGET].values.astype(np.float64)
y      = np.log1p(y_raw).astype(np.float32)

base_feature_cols = [
    c for c in train.columns
    if c not in DROP_COLS + TE_COLS + ['experience_name', 'key_skills_name', 'languages_name']
    and train[c].dtype != object
]

X_base_train = train[base_feature_cols].reset_index(drop=True)
X_base_test  = test[[c for c in base_feature_cols if c in test.columns]].reset_index(drop=True)
X_base_test  = X_base_test.reindex(columns=X_base_train.columns)

print(f'Base features: {X_base_train.shape[1]}  →  {base_feature_cols}')

Base features: 13  →  ['schedule_name', 'accept_handicapped', 'accept_kids', 'if_foreign_language', 'is_branded_description', 'employment_name', 'experience_ord', 'desc_word_count', 'has_skills', 'has_languages', 'skills_count', 'is_moscow', 'is_spb']


### 5. Smoothed Target Encoding Helper

In [5]:
def smoothed_te(
    col_tr: pd.Series,
    col_val: pd.Series,
    col_te: pd.Series,
    y_tr: np.ndarray,
    global_mean: float,
    k: int = STE_K,
) -> tuple[pd.Series, pd.Series, pd.Series]:
    """
    Smoothed TE: te(cat) = (n_cat * mean_cat + k * global_mean) / (n_cat + k)
    Shrinks small groups toward global mean, preventing overfitting on rare categories.
    """
    stats = pd.DataFrame({'y': y_tr, 'cat': col_tr.values}).groupby('cat')['y'].agg(['mean', 'count'])
    stats['smooth'] = (stats['count'] * stats['mean'] + k * global_mean) / (stats['count'] + k)
    te_map = stats['smooth']

    return (
        col_tr.map(te_map).fillna(global_mean),
        col_val.map(te_map).fillna(global_mean),
        col_te.map(te_map).fillna(global_mean),
    )


def build_fold_features(
    tr_idx: np.ndarray,
    val_idx: np.ndarray,
    y: np.ndarray,
    global_mean: float,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Assemble full feature matrix for one fold: base + SVD + smoothed TE + interaction features."""
    te_tr_list, te_val_list, te_te_list = [], [], []

    for col in TE_COLS:
        col_tr  = train[col].iloc[tr_idx].reset_index(drop=True)
        col_val = train[col].iloc[val_idx].reset_index(drop=True)
        col_te  = test[col].reset_index(drop=True)

        s_tr, s_val, s_te = smoothed_te(col_tr, col_val, col_te, y[tr_idx], global_mean)

        te_tr_list.append(s_tr.rename(f'te_{col}'))
        te_val_list.append(s_val.rename(f'te_{col}'))
        te_te_list.append(s_te.rename(f'te_{col}'))

    te_tr  = pd.concat(te_tr_list,  axis=1)
    te_val = pd.concat(te_val_list, axis=1)
    te_te  = pd.concat(te_te_list,  axis=1)

    def add_interactions(base: pd.DataFrame, te: pd.DataFrame) -> pd.DataFrame:
        out = pd.concat([base, te], axis=1)
        # experience × role salary (seniority-adjusted role signal)
        out['exp_x_role_te']    = out['experience_ord'].astype(float) * out['te_professional_roles_name']
        # moscow premium × experience
        out['moscow_x_exp']     = out['is_moscow'].astype(float) * out['experience_ord'].astype(float)
        # employer size proxy × employer salary level
        out['skills_x_lang']    = out['skills_count'].astype(float) * out['has_languages'].astype(float)
        return out

    X_tr  = add_interactions(
        pd.concat([X_base_train.iloc[tr_idx].reset_index(drop=True),  svd_train.iloc[tr_idx].reset_index(drop=True)],  axis=1), te_tr
    )
    X_val = add_interactions(
        pd.concat([X_base_train.iloc[val_idx].reset_index(drop=True), svd_train.iloc[val_idx].reset_index(drop=True)], axis=1), te_val
    )
    X_te  = add_interactions(
        pd.concat([X_base_test.reset_index(drop=True),                svd_test.reset_index(drop=True)],                axis=1), te_te
    )

    return X_tr, X_val, X_te


kf          = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
global_mean = float(np.mean(y))
print(f'global_mean (log1p scale): {global_mean:.4f}  →  salary ~{np.expm1(global_mean):.0f}')

global_mean (log1p scale): 10.6956  →  salary ~44162


### 6. LightGBM — K-Fold OOF

In [6]:
lgb_params = {
    'objective'        : 'regression',
    'metric'           : 'rmse',
    'n_estimators'     : 5000,
    'learning_rate'    : 0.02,
    'num_leaves'       : 255,
    'max_depth'        : -1,
    'min_child_samples': 15,
    'feature_fraction' : 0.6,
    'bagging_fraction' : 0.8,
    'bagging_freq'     : 5,
    'reg_alpha'        : 0.05,
    'reg_lambda'       : 0.1,
    'random_state'     : SEED,
    'n_jobs'           : -1,
    'verbose'          : -1,
}

oof_lgb  = np.zeros(len(train), dtype=np.float64)
pred_lgb = np.zeros(len(test),  dtype=np.float64)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_base_train)):
    X_tr, X_val, X_te = build_fold_features(tr_idx, val_idx, y, global_mean)

    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(
        X_tr, y[tr_idx],
        eval_set=[(X_val, y[val_idx])],
        callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(1000)],
    )

    val_preds        = np.expm1(model.predict(X_val))
    oof_lgb[val_idx] = val_preds
    pred_lgb        += np.expm1(model.predict(X_te)) / N_SPLITS

    mape = np.mean(np.abs((y_raw[val_idx] - val_preds) / (y_raw[val_idx] + 1e-8)))
    print(f'Fold {fold+1} | LGB MAPE: {mape:.4f} | iter: {model.best_iteration_}')

lgb_oof_mape = np.mean(np.abs((y_raw - oof_lgb) / (y_raw + 1e-8)))
print(f'\nLightGBM OOF MAPE: {lgb_oof_mape:.4f}')

[1000]	valid_0's rmse: 0.372715
Fold 1 | LGB MAPE: 0.2744 | iter: 1400
[1000]	valid_0's rmse: 0.36244
[2000]	valid_0's rmse: 0.361511
[3000]	valid_0's rmse: 0.361385
Fold 2 | LGB MAPE: 0.2646 | iter: 3369
[1000]	valid_0's rmse: 0.368424
[2000]	valid_0's rmse: 0.367567
Fold 3 | LGB MAPE: 0.2704 | iter: 2222
[1000]	valid_0's rmse: 0.36185
[2000]	valid_0's rmse: 0.361245
Fold 4 | LGB MAPE: 0.2661 | iter: 2006
[1000]	valid_0's rmse: 0.359623
[2000]	valid_0's rmse: 0.358799
Fold 5 | LGB MAPE: 0.2636 | iter: 2222

LightGBM OOF MAPE: 0.2679


### 7. XGBoost — K-Fold OOF

In [8]:
xgb_params = {
    'objective'            : 'reg:squarederror',
    'eval_metric'          : 'rmse',
    'n_estimators'         : 5000,
    'learning_rate'        : 0.02,
    'max_depth'            : 7,
    'min_child_weight'     : 10,
    'subsample'            : 0.8,
    'colsample_bytree'     : 0.6,
    'reg_alpha'            : 0.05,
    'reg_lambda'           : 0.1,
    'random_state'         : SEED,
    'n_jobs'               : -1,
    'tree_method'          : 'hist',
    'verbosity'            : 0,
    'early_stopping_rounds': 150,
}

oof_xgb  = np.zeros(len(train), dtype=np.float64)
pred_xgb = np.zeros(len(test),  dtype=np.float64)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_base_train)):
    X_tr, X_val, X_te = build_fold_features(tr_idx, val_idx, y, global_mean)

    model = xgb.XGBRegressor(**xgb_params)
    model.fit(
        X_tr, y[tr_idx],
        eval_set=[(X_val, y[val_idx])],
        verbose=1000,
    )

    val_preds        = np.expm1(model.predict(X_val))
    oof_xgb[val_idx] = val_preds
    pred_xgb        += np.expm1(model.predict(X_te)) / N_SPLITS

    mape = np.mean(np.abs((y_raw[val_idx] - val_preds) / (y_raw[val_idx] + 1e-8)))
    print(f'Fold {fold+1} | XGB MAPE: {mape:.4f} | iter: {model.best_iteration}')

xgb_oof_mape = np.mean(np.abs((y_raw - oof_xgb) / (y_raw + 1e-8)))
print(f'\nXGBoost OOF MAPE: {xgb_oof_mape:.4f}')

[0]	validation_0-rmse:0.52893
[1000]	validation_0-rmse:0.36530
[2000]	validation_0-rmse:0.36226
[3000]	validation_0-rmse:0.36135
[4000]	validation_0-rmse:0.36097
[4999]	validation_0-rmse:0.36082
Fold 1 | XGB MAPE: 0.2669 | iter: 4993
[0]	validation_0-rmse:0.53214
[1000]	validation_0-rmse:0.35907
[2000]	validation_0-rmse:0.35492
[3000]	validation_0-rmse:0.35348
[4000]	validation_0-rmse:0.35289
[4999]	validation_0-rmse:0.35258
Fold 2 | XGB MAPE: 0.2589 | iter: 4998
[0]	validation_0-rmse:0.53375
[1000]	validation_0-rmse:0.36647
[2000]	validation_0-rmse:0.36335
[3000]	validation_0-rmse:0.36226
[4000]	validation_0-rmse:0.36180
[4999]	validation_0-rmse:0.36166
Fold 3 | XGB MAPE: 0.2670 | iter: 4998
[0]	validation_0-rmse:0.52996
[1000]	validation_0-rmse:0.35823
[2000]	validation_0-rmse:0.35507
[3000]	validation_0-rmse:0.35404
[4000]	validation_0-rmse:0.35363
[4646]	validation_0-rmse:0.35355
Fold 4 | XGB MAPE: 0.2595 | iter: 4496
[0]	validation_0-rmse:0.52734
[1000]	validation_0-rmse:0.35733
[

### 8. CatBoost — K-Fold OOF

In [9]:
cb_params = dict(
    loss_function       = 'RMSE',
    eval_metric         = 'RMSE',
    iterations          = 5000,
    learning_rate       = 0.02,
    depth               = 7,
    l2_leaf_reg         = 3.0,
    random_strength     = 1.0,
    bagging_temperature = 0.5,
    od_type             = 'Iter',
    od_wait             = 150,
    random_seed         = SEED,
    thread_count        = -1,
    verbose             = 1000,
)

oof_cb  = np.zeros(len(train), dtype=np.float64)
pred_cb = np.zeros(len(test),  dtype=np.float64)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_base_train)):
    X_tr, X_val, X_te = build_fold_features(tr_idx, val_idx, y, global_mean)

    train_pool = Pool(X_tr,  y[tr_idx])
    val_pool   = Pool(X_val, y[val_idx])

    model = CatBoostRegressor(**cb_params)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)

    val_preds        = np.expm1(model.predict(val_pool))
    oof_cb[val_idx]  = val_preds
    pred_cb         += np.expm1(model.predict(Pool(X_te))) / N_SPLITS

    mape = np.mean(np.abs((y_raw[val_idx] - val_preds) / (y_raw[val_idx] + 1e-8)))
    print(f'Fold {fold+1} | CB MAPE: {mape:.4f}')

cb_oof_mape = np.mean(np.abs((y_raw - oof_cb) / (y_raw + 1e-8)))
print(f'\nCatBoost OOF MAPE: {cb_oof_mape:.4f}')

0:	learn: 0.5279888	test: 0.5289366	best: 0.5289366 (0)	total: 70.6ms	remaining: 5m 52s
1000:	learn: 0.2146986	test: 0.3839030	best: 0.3839030 (1000)	total: 10.4s	remaining: 41.6s
2000:	learn: 0.1878481	test: 0.3783406	best: 0.3783390 (1999)	total: 19.4s	remaining: 29s
3000:	learn: 0.1688250	test: 0.3754220	best: 0.3754169 (2996)	total: 28s	remaining: 18.6s
4000:	learn: 0.1534646	test: 0.3736642	best: 0.3736475 (3969)	total: 36.6s	remaining: 9.15s
4999:	learn: 0.1404018	test: 0.3723791	best: 0.3723752 (4994)	total: 45.2s	remaining: 0us

bestTest = 0.3723752467
bestIteration = 4994

Shrink model to first 4995 iterations.
Fold 1 | CB MAPE: 0.2815
0:	learn: 0.5270530	test: 0.5325666	best: 0.5325666 (0)	total: 11.1ms	remaining: 55.5s
1000:	learn: 0.2161987	test: 0.3747071	best: 0.3747071 (1000)	total: 8.81s	remaining: 35.2s
2000:	learn: 0.1900223	test: 0.3675165	best: 0.3675165 (2000)	total: 17.3s	remaining: 26s
3000:	learn: 0.1709389	test: 0.3638145	best: 0.3638113 (2999)	total: 26.2s	rem

### 9. Ensemble Blending (OOF-weighted)

In [10]:
w_lgb = 1.0 / lgb_oof_mape
w_xgb = 1.0 / xgb_oof_mape
w_cb  = 1.0 / cb_oof_mape
w_sum = w_lgb + w_xgb + w_cb

oof_blend  = (w_lgb * oof_lgb  + w_xgb * oof_xgb  + w_cb * oof_cb)  / w_sum
pred_blend = (w_lgb * pred_lgb + w_xgb * pred_xgb + w_cb * pred_cb) / w_sum

blend_mape = np.mean(np.abs((y_raw - oof_blend) / (y_raw + 1e-8)))
print(f'LGB  weight: {w_lgb/w_sum:.3f}  OOF MAPE: {lgb_oof_mape:.4f}')
print(f'XGB  weight: {w_xgb/w_sum:.3f}  OOF MAPE: {xgb_oof_mape:.4f}')
print(f'CB   weight: {w_cb/w_sum:.3f}  OOF MAPE: {cb_oof_mape:.4f}')
print(f'Ensemble OOF MAPE: {blend_mape:.4f}')

LGB  weight: 0.334  OOF MAPE: 0.2679
XGB  weight: 0.341  OOF MAPE: 0.2623
CB   weight: 0.324  OOF MAPE: 0.2761
Ensemble OOF MAPE: 0.2673


### 10. Post-processing & Submission

In [11]:
final_preds = np.clip(pred_blend, a_min=0, a_max=None)

test_ids = pd.read_csv('data/test_x.csv', usecols=['id'])['id']

submission = pd.DataFrame({
    'id': test_ids.values,
    'salary_mean_net': final_preds,
})

submission.to_csv('submission_v3.csv', index=False)
print(submission.head(10))
print(f'\nSubmission shape : {submission.shape}')
print(f'Negative preds   : {(final_preds < 0).sum()}')
print(f'min={final_preds.min():.0f}  median={np.median(final_preds):.0f}  max={final_preds.max():.0f}')

         id  salary_mean_net
0  46224201    114064.820858
1  42119402     46613.938993
2  45716401     28241.850419
3  43716203     38115.560197
4  47109602     66136.895603
5  45507200     37938.676735
6  41123802     67515.047653
7  49155602     42514.635315
8  41169603     24082.150275
9  48041400     49249.860732

Submission shape : (12263, 2)
Negative preds   : 0
min=11249  median=43757  max=169531
